<a id="Snowflake_Feature_Store_L1"></a>
# Snowflake Feature Store

The Snowflake Feature Store enables data engineers, data scientists and ML engineers to centralize the curation, maintenance and sharing of features that can be used in machine-learning models.

These features come from data or are the result of feature engineering, a technique where raw data is transformed into features that can be used to train machine learning models, it is a vital part of building high-quality machine learning applications. A feature store lets you easily find and employ features that work with your data.
  
"**A feature store is an emerging data system used for machine learning, serving as a centralized hub for storing, processing, and accessing commonly used features. It's making them available for reuse in the development of future machine learning models.**"

The Snowflake Feature Store is designed to help make storing and managing features for data science and machine learning workloads easier and more efficient. It provides:
- A Python SDK for defining, registering, retrieving, and managing features.

- Backend infrastructure with Snowflake tables, dynamic tables, and tags for automating feature pipelines and governance.

<img src="../../images/snowflake_feature_store.png" alt="feature_store" style="width:70%;display:block;margin-left:10%;" />

 

<a id="topics"></a>
### 1.1 Topics in this Lesson


1. [Snowflake Feature Store](#Snowflake_Feature_Store_L1)
    1. [Topics in this Lesson](#topics)
    1. [Before we start!](#Before_we_start)  
    1. [Initial setup](#Initial_setup)   
1. [Snowflake Feature Store](#Snowflake_Feature_Store)  
    1. [Feature Store entities](#Feature_Store_entities)  
1. [The Feature Store in this lecture](#The_Feature_Store_in_this_lecture)  
    1. [Use Case Introduction - Customer Segmentation](#Customer_Segmentation)  
    1. [Full setup](#Full_Setup)  
    1. [Finished setup](#Finished_setup) 

<a id="Before_we_start"></a>
### 1.2 Before we start!
If you also want to also run this demo, you need to install the latest version of snowflake_ml_python, which can be downloaded here:

> - [Snowflake_ML_Python](https://drive.google.com/drive/folders/1oyMlS-vS0-lvtG-79CFyWtGbSAkEenJ_)


In [1]:
pip install snowflake-ml-python -U


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


If the install was successful then the import of packages should not return an error

<a id="Initial_setup"></a>
### 1.3. Initial setup

For this lecture 

In [2]:
# Run utils notebook
# %run ../../utils/ds_utils_python.ipynb

# Connect to Snowflake and create a Session object named session

from snowflake.snowpark import Session

session = Session.builder.configs({
      "account":   "ES10286-ML_ENTERPRISE",
      "user":      "RKIRK",
      "password":  "8d!upvFs2#BDDB5JQ*7",
      "role":      "DEV_RK_FINANCE_SYSADMIN",
      "warehouse": "DAFT_WH",
      "database":  "RK_SANDPIT",
      "schema":    "DAFT_SCHEMA"
  }).create()

<a id="Snowflake_Feature_Store"></a>
## 2. Snowflake Feature Store

The key benefits of Snowflake’s Feature Store are:

- Easy authoring of common feature transformations in Python or SQL

- Support for batch and streaming data

- Continuous, automated, incremental feature updates on new data

- Support for backfill and point-in-time correct features with ASOF JOIN

- Fine-grained Role-based Access Control and governance

- API for retrieving features and creating training datasets

- Support for feature pipelines authored and maintained in tools such as DBT as well as in Snowflake-native pipelnes

- Integration with Model Registry and other Snowflake ML features


> **&#128221; Note:** An [ASOF JOIN](https://docs.snowflake.com/en/sql-reference/constructs/asof-join) operation combines rows from two tables based on timestamp values that follow each other, precede each other, or match exactly. For each row in the first (or left) table, the join finds a single row in the second (or right) table that has the closest timestamp value.


#### A feature store in Snowflake is a schema. 

A Snowflake Feature Store is not a unique object. It is a schema.

This enables you to create as many feature stores as you would need. 

These feature stores contain feature views, which encapsulate a pipeline for transforming raw data into one or more related features that are refreshed from the data source.

These feature views in Snowflake are Dynamic tables or Views. 

When a feature view gets materialized on a specific tables, it will use a dynamic table.

<a id="Feature_Store_entities"></a>
### 2.1 Feature Store entities

These Feature views are organized in the feature store according to the `entity` to which they apply. An entity is a higher-level abstraction that represents what the features are about. 

For example, with vending machine data, like which cans of soda someone bought, the main entities might be consumers and soda brands. In Snowflake it translates to a Tag being given to every featureView.



But an `entity` is not the only Feature Store object:

**Feature Store Back End**

Which Snowflake objects belong to which feature store objects:


- **Feature Store** -> Schema
- **Feature View**  -> Dynamic Tables (internal features) or view (external features)
- **Entity**        -> Tag
- **Feature**       -> column in a dynamic table (internal features) or view (external features)
- **FeatureSlice**  -> A FeatureSlice provides a way of creating a subset of the Features from a single FeatureViews when needed.


<a id="The_Feature_Store_in_this_lecture"></a>
## 3. The Feature Store in this lecture

**The main focus of these notebooks will be the feature store, however we will also touch on various other topics:**

-  Use stages and tables to ingest and organize raw data from S3 into Snowflake tables. 

- Leverage Snowpark's Python DataFrames to perform data cleansing, transformations such as group by, aggregate, pivot, and join to create features for machine learning.

- Use Snowflake's Feature Store to register and maintain feature-engineering pipelines and understand how to monitor them once operational.

- Perform feature transformation and run ML Training in Snowflake using Snowpark ML. Register the trained ML model for inference from the Snowflake Model Registry

- Implementing a production inference data-pipeline to maintain customer segments as underlying customer behaviors change in source data.

The diagram below provides an overview of what we will be building in this lecture:

<img src="../../images/feature_store_lecture_overview.png" alt="feature_store_lecture_diagram" style="width:90%;display:block;margin-left:10%;" />

<a id="Customer_Segmentation"></a>
### 3.1 Use Case Introduction - Customer Segmentation

This Segmentation use case is designed to emulate a data science pipeline to find clusters of customers based on aggregate features where the customers are grouped based on their spending behavior.

It involves creating subgroups of customers based on similar traits.

The input in this use case consists of order and return transaction data from a retail business.

The use case uses Tables Customer, Order, Lineitem and Order_returns.

K-means clustering algorithm is used to derive the optimum number of clusters and understand the underlying customer segments based on the data provided.

We will use the Use-Case to show how Snowflake Feature Store (and Model Registry) can be used to maintain & store features, retrieve them for training and perform micro-batch inference.

In the development (TRAINING) enviroment we will

- create FeatureViews in the Feature Store that maintain the required customer-behaviour features.
- use these Features to train a model, and save the model in the Snowflake model-registry.
- plot the clusters for the trained model to visually verify.

In the production (SERVING) environment we will

- re-create the FeatureViews on production data
- generate an Inference FeatureView that uses the saved model to perform incremental inference



<a id="Full_Setup"></a>
### 3.2 Full setup

- We will now need to setup the Databases to mimic development and production database environments.

- We will also create something that will mimic regular data ingestion from source systems into our production environment. 

- The dataset that we will use is the TPCX-AI dataset. Which is a artificial dataset created for ML and AI.

We will do the following below:
    
Roles & Permissions
- We create a Data Scientist role
- Give the role access to the Databases

Databases
- We create 2 databases
    - SEGMENTATION - a database containing static data and that holds 3 schemas
        - Training
        - Scoring
        - Serving
    - SEGMENTATION_LIVE - Same schemas but also a config schema. It will incrementally ingest data from SEGMENTATION to mimic a live environment
        - Training
        - Scoring
        - Serving
        - Config - Containing File formats, tasks, streams etc


> **&#128221; Note:** *You needn't edit anything in the following cells. Just understand what they do and run it.*


In [3]:
# Create an empty DataFrame for reflecting upon our context items
context_df = session.create_dataframe([""]).to_df("")

# Retrieve the current username
from snowflake.snowpark.functions import current_user
username = (str(context_df
    .select(current_user())
    .collect()[0][0]
   )
)
print(f"The current user is: {username}")  # NB we dont need this - tutorial used it to prefix some object names.

# Roles
aa_role = 'training_role'        
ds_role = 'DS_ROLE'             # The Data-Scientist role that will create and have permissions over the data, Feature-Store and Model-Registry

# Database
segmentation_database_base       = f'SEGMENTATION' # The Database we will create to contain the base static data
segmentation_database_live        = f'{segmentation_database_base}_LIVE' # The Database we will create to contain the pseudo 'Live' incrementing data
databases = [segmentation_database_base,segmentation_database_live]


# Schemas
segmentation_config_schema       = 'CONFIG'   # This Config Schema containing database artifacts
segmentation_training_schema     = 'TRAINING' # The Training (Development) schema
segmentation_scoring_schema      = 'SCORING'  # The Scoring (Test) schema
segmentation_serving_schema      = 'SERVING'  # The Serving (Production) schema
schemas = [segmentation_training_schema, segmentation_scoring_schema, segmentation_serving_schema,]
fq_schemas = []
for d in databases:
    for s in schemas:
        fq_schemas.append(f'''{d}.{s}''')        

# Stage
segmentation_internal_stage = f'SEGMENTATION_STAGE'  # The Stage name we will use to represent the S3 bucket

# Warehouse
segmentation_warehouse = f'SEGMENTATION_WH'  # The name of the Warehouse we will use
initial_wh_size = 'XSMALL' 

The current user is: RKIRK


In [4]:
# Can query variables just like this:
fq_schemas

['SEGMENTATION.TRAINING',
 'SEGMENTATION.SCORING',
 'SEGMENTATION.SERVING',
 'SEGMENTATION_LIVE.TRAINING',
 'SEGMENTATION_LIVE.SCORING',
 'SEGMENTATION_LIVE.SERVING']

#### Block 1 -  Create data scientist role and a warehouse, and give permissions to the data scientist role

In [12]:
# Use training_role
session.sql(f"use role ACCOUNTADMIN").collect() # NB I chose this role - tutorial used specific training role 

# Role
session.sql(f"create role if not exists {ds_role}").collect()
session.sql(f"grant role {ds_role} to role SYSADMIN").collect()

# # Warehouse
session.sql(f"create warehouse if not exists {segmentation_warehouse}  warehouse_size = {initial_wh_size}").collect()
session.sql(f"grant all on warehouse {segmentation_warehouse} to role {ds_role}").collect()
session.sql(f"use warehouse {segmentation_warehouse}").collect()

# # Tasks
session.sql(f"grant execute managed task on account to role {ds_role}").collect()
session.sql(f"grant execute task on account to role {ds_role}").collect()

[Row(status='Statement executed successfully.')]

#### Block 2 - Create databases and grant database permissions to the data scientist role

In [13]:
session.sql(f"use role ACCOUNTADMIN").collect() # NB I chose this role - tutorial used specific training role 
session.sql(f"grant create database on account to role {ds_role}").collect()

session.sql(f"create or replace database {segmentation_database_base}").collect()
session.sql(f"grant all on database {segmentation_database_base} to role {ds_role}").collect()
session.sql(f"grant all on all schemas in database {segmentation_database_base} to role {ds_role}").collect()
session.sql(f"grant all on future schemas in database {segmentation_database_base} to role {ds_role}").collect()

session.sql(f"create or replace database {segmentation_database_live}").collect()
session.sql(f"grant all on database {segmentation_database_live} to role {ds_role}").collect()
session.sql(f"grant all on all schemas in database {segmentation_database_live} to role {ds_role}").collect()
session.sql(f"grant all on future schemas in database {segmentation_database_live} to role {ds_role}").collect()

[Row(status='Statement executed successfully.')]

MyNotes: The above creates databases SEGMENTATION and SEGMENTATION_LIVE
I did this on BIC ML_ENTERPRISE account 

#### Block 3 - Create objects within the SEGMENTATION database

In [ ]:
session.sql(f"use role {ds_role}").collect()

session.sql(f"create schema if not exists {segmentation_database_base}.{segmentation_training_schema}").collect()
session.sql(f"use schema {segmentation_database_base}.{segmentation_training_schema}").collect()

session.sql(f"create schema  if not exists {segmentation_database_base}.{segmentation_serving_schema}").collect()
session.sql(f"use schema {segmentation_database_base}.{segmentation_serving_schema}").collect()

session.sql(f"create schema  if not exists {segmentation_database_base}.{segmentation_scoring_schema}").collect()
session.sql(f"use schema {segmentation_database_base}.{segmentation_scoring_schema}").collect()

session.sql(f"create schema  if not exists {segmentation_database_base}.{segmentation_config_schema}").collect()
session.sql(f"use schema {segmentation_database_base}.{segmentation_config_schema}").collect()

session.sql(f"create file format if not exists {segmentation_database_base}.{segmentation_config_schema}.parquet_ff type = 'parquet' ").collect()

# MyNote: This creates an external stage that points to Snowflake's public S3 quickstart bucket which contains the data files.
session.sql(f"""create stage if not exists {segmentation_database_base}.{segmentation_config_schema}.{segmentation_internal_stage} 
        file_format = {segmentation_database_base}.{segmentation_config_schema}.parquet_ff 
        url = 's3://sfquickstarts/getting_started_with_snowflake_feature_store/' """).collect() # MyNote: this parameter is what distinguishes it as an external stage

[Row(status='SEGMENTATION_STAGE already exists, statement succeeded.')]

#### Block 4 - Loading data into the SEGMENTATION database using COPY INTO

In [16]:
session.sql(f"use role {ds_role}").collect()

# Calculate the DATE point difference between the source data and todays date.  
# This reference point will be used to select a subset of the data for pre-loading, and the remainder will be incrementally ingested via a scheduled task.
date_diff_to_source = session.sql('''select timestampdiff('days',  '2013-04-01', CURRENT_DATE() )::VARCHAR date_diff_to_source''').collect()[0][0]
print('Difference in Days between source data and current date :',date_diff_to_source)


# Iteration over the Schemas creating and loading the required tables in each
for s in [segmentation_training_schema, segmentation_serving_schema, segmentation_scoring_schema]:

    # CUSTOMER
    session.sql(f"""CREATE OR REPLACE TABLE {segmentation_database_base}.{s}.CUSTOMER
                  (C_CUSTOMER_SK INTEGER,
                   C_CUSTOMER_ID VARCHAR,
                   C_CURRENT_ADDR_SK INTEGER,
                   C_FIRST_NAME VARCHAR,
                   C_LAST_NAME VARCHAR,
                   C_PREFERRED_CUST_FLAG VARCHAR,
                   C_BIRTH_DAY INTEGER,
                   C_BIRTH_MONTH INTEGER,
                   C_BIRTH_YEAR INTEGER,
                   C_BIRTH_COUNTRY VARCHAR,
                   C_LOGIN VARCHAR,
                   C_EMAIL_ADDRESS VARCHAR,
                   C_CLUSTER_ID INTEGER
                  ) CLUSTER BY (C_CUSTOMER_SK);
    """).collect()
    session.sql(f"""COPY INTO {segmentation_database_base}.{s}.CUSTOMER 
            FROM 
                 (select $1:C_CUSTOMER_SK::INTEGER,
                         $1:C_CUSTOMER_ID::VARCHAR,
                         $1:C_CURRENT_ADDR_SK::INTEGER,
                         $1:C_FIRST_NAME::VARCHAR,
                         $1:C_LAST_NAME::VARCHAR,
                         $1:C_PREFERRED_CUST_FLAG::VARCHAR,
                         $1:C_BIRTH_DAY::INTEGER,
                         $1:C_BIRTH_MONTH::INTEGER,
                         $1:C_BIRTH_YEAR::INTEGER,
                         $1:C_BIRTH_COUNTRY::VARCHAR,
                         $1:C_LOGIN::VARCHAR,
                         $1:C_EMAIL_ADDRESS::VARCHAR,                        
                         $1:C_CLUSTER_ID::INTEGER
                  from  @{segmentation_database_base}.CONFIG.{segmentation_internal_stage}/{s}/CUSTOMER) 
            FILE_FORMAT = (FORMAT_NAME = '{segmentation_database_base}.{segmentation_config_schema}.parquet_ff' ) """).collect() 
    
    # ORDERS
    session.sql(f"""CREATE OR REPLACE TABLE {segmentation_database_base}.{s}.ORDERS
    (O_ORDER_ID INTEGER,
     O_CUSTOMER_SK INTEGER,
     ORDER_TS TIMESTAMP,
     WEEKDAY VARCHAR,
     ORDER_DATE DATE,
     STORE INTEGER,
     TRIP_TYPE INTEGER)
     CLUSTER BY (O_ORDER_ID, ORDER_DATE)
    """).collect()
    session.sql(f"""COPY INTO {segmentation_database_base}.{s}.ORDERS 
            FROM 
                 (select $1:O_ORDER_ID::INTEGER,
                         $1:O_CUSTOMER_SK::INTEGER,
                         timestampadd('MINS', UNIFORM( -1440 , 0 , random() ) ,timestampadd('days',   {date_diff_to_source}, $1:"DATE"::DATE)) ORDER_TS,
                         decode(extract(dayofweek from ORDER_TS), 1, 'Monday', 2, 'Tuesday', 3, 'Wednesday', 4, 'Thursday',  5, 'Friday',  6, 'Saturday',  0, 'Sunday') WEEKDAY,
                         TO_DATE(ORDER_TS) ORDER_DATE,
                         $1:STORE::INTEGER,
                         $1:TRIP_TYPE::INTEGER
                  from  @{segmentation_database_base}.CONFIG.{segmentation_internal_stage}/{s}/ORDERS) 
            FILE_FORMAT = (FORMAT_NAME = '{segmentation_database_base}.{segmentation_config_schema}.parquet_ff' ) """).collect() 
    
    # LINEITEM
    session.sql(f"""CREATE OR REPLACE TABLE {segmentation_database_base}.{s}.LINEITEM
    (LI_ORDER_ID INTEGER,
     LI_PRODUCT_ID INTEGER,
     QUANTITY INTEGER,
     PRICE DECIMAL(8,2))
    CLUSTER BY (LI_PRODUCT_ID, LI_ORDER_ID)
    """).collect()
    session.sql(f"""COPY INTO {segmentation_database_base}.{s}.LINEITEM 
            FROM (select $1:LI_ORDER_ID::INTEGER,
                         $1:LI_PRODUCT_ID::INTEGER,
                         $1:QUANTITY::INTEGER,
                         $1:PRICE::DECIMAL(8,2)
                  from @{segmentation_database_base}.CONFIG.{segmentation_internal_stage}/{s}/LINEITEM) 
            FILE_FORMAT = (FORMAT_NAME = '{segmentation_database_base}.{segmentation_config_schema}.parquet_ff' ) """).collect() 
    
    # ORDER_RETURNS
    session.sql(f"""CREATE OR REPLACE TABLE {segmentation_database_base}.{s}.ORDER_RETURNS
    (OR_ORDER_ID INTEGER,
     OR_PRODUCT_ID INTEGER,
     OR_RETURN_QUANTITY INTEGER)
     CLUSTER BY (OR_PRODUCT_ID, OR_ORDER_ID);
    """).collect()
    session.sql(f"""COPY INTO {segmentation_database_base}.{s}.ORDER_RETURNS 
            FROM (select $1:OR_ORDER_ID::INTEGER,
                         $1:OR_PRODUCT_ID::INTEGER,
                         $1:OR_RETURN_QUANTITY::INTEGER
                  from  @{segmentation_database_base}.CONFIG.{segmentation_internal_stage}/{s}/ORDER_RETURNS) 
            FILE_FORMAT = (FORMAT_NAME = '{segmentation_database_base}.{segmentation_config_schema}.parquet_ff' ) """).collect() 

print ('done')

Difference in Days between source data and current date : 4793
done


MyNotes: The above creates tables and loads data into them from the external stage - it creates the tables on all the schemas.
NB There is separate data file for data for each schema;

```sql
select count(1) from SEGMENTATION.SCORING.CUSTOMER;
--10000
select count(1) from SEGMENTATION.SERVING.CUSTOMER;
--7071
select count(1) from SEGMENTATION.TRAINING.CUSTOMER;
--70710
```

<img src="../../images/my-images/tables-for-features.png" alt="tables-for-features" style="width:15%;display:block;margin-left:10%;" />

#### Block 5 - Creating schemas within the SEGMENTATION_LIVE database and giving the data scientist role permissions

In [17]:
# Training schema
session.sql(f"""use role {ds_role}""").collect()
session.sql(f"""create schema if not exists {segmentation_database_live}.{segmentation_training_schema}""").collect()
session.sql(f"""grant usage on schema {segmentation_database_live}.{segmentation_training_schema} to role {ds_role}""").collect()
session.sql(f"""grant create table on schema {segmentation_database_live}.{segmentation_training_schema} to role {ds_role}""").collect()
session.sql(f"""grant create view on schema {segmentation_database_live}.{segmentation_training_schema} to role {ds_role}""").collect()
session.sql(f"""grant create tag on schema {segmentation_database_live}.{segmentation_training_schema} to role {ds_role}""").collect()
session.sql(f"""grant create dataset on schema {segmentation_database_live}.{segmentation_training_schema} to {ds_role}""").collect()
session.sql(f"""grant select,references on all views in schema {segmentation_database_live}.{segmentation_training_schema} to role {ds_role}""").collect()
session.sql(f"""grant create dynamic table on schema {segmentation_database_live}.{segmentation_training_schema} to role {ds_role}""").collect()
session.sql(f"""grant select,monitor on all dynamic tables in schema {segmentation_database_live}.{segmentation_training_schema} to role {ds_role}""").collect()
session.sql(f"""grant usage on all datasets in schema {segmentation_database_live}.{segmentation_training_schema} to role {ds_role}""").collect()

# Serving schema
session.sql(f"""use role {ds_role}""").collect()
session.sql(f"""create schema if not exists {segmentation_database_live}.{segmentation_serving_schema}""").collect()
session.sql(f"""grant usage on schema {segmentation_database_live}.{segmentation_serving_schema} to role {ds_role}""").collect()
session.sql(f"""grant create table on schema {segmentation_database_live}.{segmentation_serving_schema} to role {ds_role}""").collect()
session.sql(f"""grant create view on schema {segmentation_database_live}.{segmentation_serving_schema} to role {ds_role}""").collect()
session.sql(f"""grant create tag on schema {segmentation_database_live}.{segmentation_serving_schema} to role {ds_role}""").collect()
session.sql(f"""grant create dataset on schema {segmentation_database_live}.{segmentation_serving_schema} to {ds_role}""").collect()
session.sql(f"""grant select,references on all views in schema {segmentation_database_live}.{segmentation_serving_schema} to role {ds_role}""").collect()
session.sql(f"""grant create dynamic table on schema {segmentation_database_live}.{segmentation_serving_schema} to role {ds_role}""").collect()
session.sql(f"""grant select,monitor on all dynamic tables in schema {segmentation_database_live}.{segmentation_serving_schema} to role {ds_role}""").collect()
session.sql(f"""grant usage on all datasets in schema {segmentation_database_live}.{segmentation_serving_schema} to role {ds_role}""").collect()

# Scoring schema
session.sql(f"""use role {ds_role}""").collect()
session.sql(f"""create schema if not exists {segmentation_database_live}.{segmentation_scoring_schema}""").collect()
session.sql(f"""grant usage on schema {segmentation_database_live}.{segmentation_scoring_schema} to role {ds_role}""").collect()
session.sql(f"""grant create table on schema {segmentation_database_live}.{segmentation_scoring_schema} to role {ds_role}""").collect()
session.sql(f"""grant create view on schema {segmentation_database_live}.{segmentation_scoring_schema} to role {ds_role}""").collect()
session.sql(f"""grant create tag on schema {segmentation_database_live}.{segmentation_scoring_schema} to role {ds_role}""").collect()
session.sql(f"""grant create dataset on schema {segmentation_database_live}.{segmentation_scoring_schema} to {ds_role}""").collect()
session.sql(f"""grant select,references on all views in schema {segmentation_database_live}.{segmentation_scoring_schema} to role {ds_role}""").collect()
session.sql(f"""grant create dynamic table on schema {segmentation_database_live}.{segmentation_scoring_schema} to role {ds_role}""").collect()
session.sql(f"""grant select,monitor on all dynamic tables in schema {segmentation_database_live}.{segmentation_scoring_schema} to role {ds_role}""").collect()
session.sql(f"""grant usage on all datasets in schema {segmentation_database_live}.{segmentation_scoring_schema} to role {ds_role}""").collect()

[Row(status='Statement executed successfully. 0 objects affected.')]

#### Block 6 - Create tables in and copying CUSTOMER data into SEGMENTATION_LIVE database. Also creating tasks and streams
- One task will load data into the ORDER table in the SEGMENTATION_LIVE database. 

- 2 tasks (using streams) will then pick that up and insert data into LINEITEM and ORDER_RETURNS

In [18]:
session.sql(f"""use role {ds_role}""").collect()
# Set up Incremental SERVING & SCORING data maintenance
for s in [segmentation_serving_schema, segmentation_scoring_schema]:

    session.sql(f"""CREATE OR REPLACE TABLE {segmentation_database_live}.{s}.CUSTOMER
                  (C_CUSTOMER_SK INTEGER,
                   C_CUSTOMER_ID VARCHAR,
                   C_CURRENT_ADDR_SK INTEGER,
                   C_FIRST_NAME VARCHAR,
                   C_LAST_NAME VARCHAR,
                   C_PREFERRED_CUST_FLAG VARCHAR,
                   C_BIRTH_DAY INTEGER,
                   C_BIRTH_MONTH INTEGER,
                   C_BIRTH_YEAR INTEGER,
                   C_BIRTH_COUNTRY VARCHAR,
                   C_LOGIN VARCHAR,
                   C_EMAIL_ADDRESS VARCHAR,
                   C_CLUSTER_ID INTEGER
                  ) CLUSTER BY (C_CUSTOMER_SK)
    """).collect()

    session.sql(f"""insert into {segmentation_database_live}.{s}.CUSTOMER select * from {segmentation_database_base}.{s}.CUSTOMER order by C_CUSTOMER_SK """).collect()
    
    session.sql(f"""CREATE OR REPLACE TABLE {segmentation_database_live}.{s}.ORDERS
    (O_ORDER_ID INTEGER,
     O_CUSTOMER_SK INTEGER,
     ORDER_TS TIMESTAMP,
     WEEKDAY VARCHAR,
     ORDER_DATE DATE,
     STORE INTEGER,
     TRIP_TYPE INTEGER)
     CLUSTER BY (O_ORDER_ID, ORDER_TS)
    """).collect()

    session.sql(f"""CREATE OR REPLACE TABLE {segmentation_database_live}.{s}.LINEITEM
    (LI_ORDER_ID INTEGER,
     LI_PRODUCT_ID INTEGER,
     QUANTITY INTEGER,
     PRICE DECIMAL(8,2))
    CLUSTER BY (LI_PRODUCT_ID, LI_ORDER_ID)
    """).collect()

    session.sql(f"""CREATE OR REPLACE TABLE {segmentation_database_live}.{s}.ORDER_RETURNS
    (OR_ORDER_ID INTEGER,
     OR_PRODUCT_ID INTEGER,
     OR_RETURN_QUANTITY INTEGER)
     CLUSTER BY (OR_PRODUCT_ID, OR_ORDER_ID)
    """).collect()

    # Streams
    session.sql(f""" create or replace stream {segmentation_database_base}.{segmentation_config_schema}.{s}_ORDER_LINEITEM_STREAM on table {segmentation_database_live}.{s}.ORDERS""").collect()
    session.sql(f""" create or replace stream {segmentation_database_base}.{segmentation_config_schema}.{s}_ORDER_ORDERRETURNS_STREAM on table {segmentation_database_live}.{s}.ORDERS""").collect() 

    session.sql(f""" create or replace task {segmentation_database_base}.{segmentation_config_schema}.APPEND_{s}_LINEITEM_TASK
    schedule='1 MINUTE'
	USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE='XSMALL'
	when SYSTEM$STREAM_HAS_DATA('{segmentation_database_base}.{segmentation_config_schema}.{s}_ORDER_LINEITEM_STREAM')
	as insert into {segmentation_database_live}.{s}.LINEITEM
select l.* 
from  {segmentation_database_base}.{s}.LINEITEM l,
      {segmentation_database_base}.{segmentation_config_schema}.{s}_ORDER_LINEITEM_STREAM o
where l.LI_ORDER_ID = o.O_ORDER_ID
order by LI_ORDER_ID, LI_PRODUCT_ID""").collect()

    session.sql(f""" alter task {segmentation_database_base}.{segmentation_config_schema}.APPEND_{s}_LINEITEM_TASK resume""").collect()

    session.sql(f""" create or replace task {segmentation_database_base}.{segmentation_config_schema}.APPEND_{s}_ORDER_RETURNS_TASK
	schedule='1 MINUTE'
	USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE='XSMALL'
	when SYSTEM$STREAM_HAS_DATA('{segmentation_database_base}.CONFIG.{s}_ORDER_ORDERRETURNS_STREAM')
	as insert into {segmentation_database_live}.{s}.ORDER_RETURNS
select o_r.* 
from {segmentation_database_base}.{s}.ORDER_RETURNS o_r,
     {segmentation_database_base}.{segmentation_config_schema}.{s}_ORDER_ORDERRETURNS_STREAM o
where o_r.OR_ORDER_ID = o.O_ORDER_ID
order by OR_ORDER_ID, OR_PRODUCT_ID """).collect()

    session.sql(f""" alter task {segmentation_database_base}.{segmentation_config_schema}.APPEND_{s}_ORDER_RETURNS_TASK resume """).collect()

    session.sql(f""" insert into {segmentation_database_live}.{s}.ORDERS
select * from {segmentation_database_base}.{s}.ORDERS o
where o.ORDER_TS < current_timestamp() 
order by ORDER_TS, O_CUSTOMER_SK """).collect()

    session.sql(f""" create or replace task {segmentation_database_base}.{segmentation_config_schema}.APPEND_{s}_ORDER_TASK
	schedule='1 MINUTE'
	USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE='XSMALL'
	as insert into {segmentation_database_live}.{s}.ORDERS
with o_max_timestamp as ( select max(ORDER_TS) max_ts
                           from {segmentation_database_live}.{s}.ORDERS )
     select O_ORDER_ID, O_CUSTOMER_SK, ORDER_TS, WEEKDAY, ORDER_DATE, STORE, TRIP_TYPE
       from {segmentation_database_base}.{s}.ORDERS o,
            o_max_timestamp fmt
      where 
            o.ORDER_TS <= current_timestamp() 
        and o.ORDER_TS > fmt.max_ts
   order by ORDER_TS, O_CUSTOMER_SK """).collect()

    session.sql(f""" alter task {segmentation_database_base}.{segmentation_config_schema}.APPEND_{s}_ORDER_TASK resume """).collect()

MyNote: Think the tasks and streams created above load data from the base db into live db - from SEGMENTATION into SEGMENTATION_LIVE:
```sql
USE ROLE ds_role;

SHOW STREAMS IN DATABASE SEGMENTATION;
SHOW STREAMS IN SCHEMA SEGMENTATION.CONFIG;

SHOW TASKS IN DATABASE SEGMENTATION;
SHOW TASKS IN SCHEMA SEGMENTATION.CONFIG;
```

#### Block 7 - Create tables in TRAINING that hold static data


In [ ]:

session.sql(f""" create OR REPLACE table {segmentation_database_live}.TRAINING.CUSTOMER      as select * from {segmentation_database_base}.TRAINING.CUSTOMER order by C_CUSTOMER_SK """).collect()
session.sql(f""" create OR REPLACE table {segmentation_database_live}.TRAINING.LINEITEM      as select * from {segmentation_database_base}.TRAINING.LINEITEM order by LI_ORDER_ID """).collect()
session.sql(f""" create OR REPLACE table {segmentation_database_live}.TRAINING.ORDERS        as select * from {segmentation_database_base}.TRAINING.ORDERS order by ORDER_TS, O_ORDER_ID, O_CUSTOMER_SK """).collect()
session.sql(f""" create OR REPLACE table {segmentation_database_live}.TRAINING.ORDER_RETURNS as select * from {segmentation_database_base}.TRAINING.ORDER_RETURNS order by OR_ORDER_ID, OR_PRODUCT_ID """).collect()

MyNote: For some reason the above hung when trying to run from notebook. I ran the SQL directly on snowsight which ran in just a few seconds;
```sql
create OR REPLACE table SEGMENTATION_LIVE.TRAINING.CUSTOMER as select * from SEGMENTATION.TRAINING.CUSTOMER order by C_CUSTOMER_SK;
create OR REPLACE table SEGMENTATION_LIVE.TRAINING.LINEITEM      as select * from SEGMENTATION.TRAINING.LINEITEM order by LI_ORDER_ID; 
create OR REPLACE table SEGMENTATION_LIVE.TRAINING.ORDERS        as select * from SEGMENTATION.TRAINING.ORDERS order by ORDER_TS, O_ORDER_ID, O_CUSTOMER_SK;
create OR REPLACE table SEGMENTATION_LIVE.TRAINING.ORDER_RETURNS as select * from SEGMENTATION.TRAINING.ORDER_RETURNS order by OR_ORDER_ID, OR_PRODUCT_ID;
```

<a id="Finished_setup"></a>
### 3.3 Finished setup
Now we have created all necessary objects. 

**1. Objects created**

There are now 2 databases created. 
- SEGMENTATION
- SEGMENTATION_LIVE

Both the databases have the schemas:
- Training
- Scoring
- Serving

And SEGMENTATION_LIVE also has the schema Config, which contains the tasks, streams and file formats.

**2. Data loaded**

In block 4 a `copy into` command copies data from a Stage into CUSTOMER, ORDERS, LINEITEMS and ORDER_RETURNS in the SEGMENTATION database.

In Block 6 the values of the CUSTOMER table from SEGMENTATION are copied into the CUSTOMER table in the SEGMENTATION_LIVE database.

Also in Block 6, 2 Streams and 3 Tasks are created.

The Tasks are created in the CONFIG schema and once per minute load data into the SEGMENTATION_LIVE database.
- A simple tasks copies data from SEGMENTATION.SERVING.ORDERS into SEGMENTATION_LIVE.SERVING.ORDERS
- A simple tasks copies data from SEGMENTATION.SCORING.ORDERS into SEGMENTATION_LIVE.SCORING.ORDERS

However, there are several other tasks created in Block 6 that trigger whenever data gets copied into the ORDERS TABLE. 

Example inline task script:

```
when SYSTEM$STREAM_HAS_DATA('SEGMENTATION.CONFIG.SERVING_ORDER_LINEITEM_STREAM')
	as insert into SEGMENTATION_LIVE.SERVING.LINEITEM
select l.* 
from  SEGMENTATION.SERVING.LINEITEM l,
      SEGMENTATION.CONFIG.SERVING_ORDER_LINEITEM_STREAM o
where l.LI_ORDER_ID = o.O_ORDER_ID
order by LI_ORDER_ID, LI_PRODUCT_ID
```

This is to mimic a real life scenario. We can see more information about this in the Task History in Snowsight